In [1]:
import pickle
import os
import numpy as np
import pandas as pd

from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.models.ctm import CombinedTM
from sentence_transformers import SentenceTransformer

In [9]:
########## TEST ############
df = pd.read_csv("../data/preprocessed/reviews_trust_clean.csv")

qualite_keywords = [
    r"cass(é|ée|és|ées)?",
    r"casse(r)?",
    r"d[ée]f(ec|aut)(s)?",
    r"qualit(é|e|és|es|ées|ees)?",
    r"mati[èe]re(s)?",
    r"soli(de|d|de)?",
    r"résistan(t|ce|ces|t)?",
    r"correspond(ait|s|t)?",
    r"fonctionne(r)?",
    r"d[ée]f(ec)?tueu(se|s|e|x|ses)?",
    r"tiss(u|us|u)?(x)?",
    r"couleur(s|e)?",
    r"((trop|un peu|très)\s+petit(e|es)?|taille(nt)?\s+(trop|un peu|très)?\s*petit(e|es)?)"
    r"grand(e|es|s)?",
    r"ray(é|ee|ée|e|és|és|ees|ée|ure|ures)",
    r"finitions?",
    r"beau(x)?",
    r"belle(s)?",
    r"joli(e|es|s)?",
    r"conforme(s)?",
    r"en panne(s)?",
    r"bas de gamme",
    r"inutilisable(s)?",
]


livraison_keywords = [
    r"livr(é|er|aison|aisons|ées|ée|eur)?",
    r"passage(s)?",
    r"coli(s)?",
    r"paquet(s)?",
    r"carton(s)?",
    r"retard(é|ée|s)?",
    r"dpd",
    r"gls",
    r"poste",
    r"facteur(s)?",
    r"mondial relay",
    r"mondial relais",
    r"relais",
    r"relay",
    r"point relais",
    r"domicile",
    r"retrait",
    r"bureau de poste",
    r"ouvert(s|e|es)?",
    r"endommag(é|ée|és|ées)?",
    r"déposé(e|s)?",
    r"laisser?|laissé(e|s)?",
    r"transporteur(s)?",
    r"manquant(e|s)?",
    r"incomplet(s|es)?",
    r"abîmé(e|s)?",
    r"déchiré(e|s)?",
]

client_keywords = [
    r"service client(s)?",
    r"\bsav\b",
    r"service après vente",
    r"répond(re|s|u)?",
    r"réponse(s)?",
    r"rembour(s|ser|sement|sez)?",
    r"contact(er|e|es)?",
    r"mail(s)?|email(s)?",
    r"appel(er|s)?",
    r"incompétent(e|s|es)?",
    r"incompétence",
    r"escroc(s)?",
    r"arnaque(s)?",
    r"voleur(s|ses)?",
    r"solution(s)?",
    r"aucune réponse",
    r"réclamation(s)?",
    r"annul(é|ée|er|ation|ations)",
    r"litige(s)?",
]

def make_regex(words):
    return r"\b(" + "|".join(words) + r")\b"

df["Qualite_Produit"] = df["clean_comment"].str.contains(make_regex(qualite_keywords), case=False, na=False).astype(int)
df["Service_Livraison"] = df["clean_comment"].str.contains(make_regex(livraison_keywords), case=False, na=False).astype(int)
df["Service_Client"] = df["clean_comment"].str.contains(make_regex(client_keywords), case=False, na=False).astype(int)

df[["Qualite_Produit", "Service_Livraison", "Service_Client"]].sum()

df_annotated = df[
    (df["Qualite_Produit"] == 1)
    | (df["Service_Livraison"] == 1)
    | (df["Service_Client"] == 1)
]

#pour afficher le nombre d'avis qui ont au moins 1 label
df_annotated.shape[0]

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

df_sample = df_annotated[["clean_comment", "Qualite_Produit", "Service_Livraison", "Service_Client"]].head(100)
df_sample

C:\Users\georg\AppData\Local\Temp\ipykernel_25900\2518547857.py:86: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Qualite_Produit"] = df["clean_comment"].str.contains(make_regex(qualite_keywords), case=False, na=False).astype(int)
C:\Users\georg\AppData\Local\Temp\ipykernel_25900\2518547857.py:87: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Service_Livraison"] = df["clean_comment"].str.contains(make_regex(livraison_keywords), case=False, na=False).astype(int)
C:\Users\georg\AppData\Local\Temp\ipykernel_25900\2518547857.py:88: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Service_Client"] = df["clean_comment"].str.contains(make_regex(client_keywords), case=False, na=False).astype(int)


,clean_comment,Qualite_Produit,Service_Livraison,Service_Client
0,"bonjour , ca doit faire 5 ans environ que je suis membre showrooprive . je n ’ ai jamais eu de soucis en particulier à part 2/3 petites bricoles…par contre depuis ces 3 derniers mois , une vraie catastrophe ! j ’ ai eu 3 commandes annulées d ’ affiliés… à se demander si les ventes sont bien réelles histoires de nous faire payer et que showroom « joue » avec notre argent en se faisant des intérêts bancaires sur notre dos ! ! ! ! la 1 ère commande « machines et cafés » 89,99€… annulé 1 mois après …la 2e commande « don ’ t call me jennifer » 65,16€… annulé 1 mois après …la 3e commande « techwood » 46,29€… annulé 1 mois après….je l ’ apprends toujours par mail…le même message « bateau » « malgré tous les efforts… .. bla-bla-bla bien évidemment , pas de dédommagement ! ! ! ! ben . non , hein , pourquoi faire ? ! ? ! je doute que ce site soit aussi fiable et sérieux qu ’ au début ! au final je vais juste rester fidèle sur n autre site marchand ! ! ! ! ! très décevant … plus du tout confiance je ne recommande plus du tout ce site et j ’ en fais part à mon entourage et réseaux sociaux ! ! afin d ’ éviter que d ’ autres personnes connaissent ce même type d ’ expérience ! !",0,0,1
1,vente lacoste article manquant photo prise sur 6 articles la moitié livrée sans explication manque de respect.numero de commande 230130353,0,1,0
2,"vente lacoste honteuse , article erroné , article manquant , pas de bon de livraison , retard et service client désastreux.après deux appels désastreux , je tenais à avertir les plus courageux qui oseront commander et contacter ce service client . showroom fait dans le pire du pire . attentions aux oreilles , votre interlocuteur vous parle dans environnement très bruyant de type chantier , coup de marteaux et jet de matériaux à proximité du micro . vous demandez de faire cesser la nuisance , on vous répond que non car ils vous entendent très bien . erreur sur votre commande , l ’ interlocuteur ne vous crois pas , il faut envoyer les captures d ’ écran de sa commande pour les convaincre qu ’ ils font bien n ’ importe quoi.la commande arrive en plusieurs fois pour une même vente , retard , article manquant et pas de bon livraison pour comprendre quoi que ce soit , on ne sait rien et il est impossible d ’ obtenir le suivi de livraison et les détails pour les articles manquants.un article annulé par showroom car n ’ était plus en stock mais toujours proposé à la vente . ( commande n°230077467 et commande n•230077467 ) ca fait beaucoup d ’ un coup , c ’ est inadmissible.fuyez !",0,1,1
4,commande téléphone etat a+ . livraison d un vieux téléphone pourri sans batterie rayé partout et inaudible ! ! ! super l affaire 300 euros a la poubelle merci showroomprivee ! ! ! passez votre chemin ils sont nuls et ne contrôlent même pas ce qu ils proposent a la vente .,1,1,0
5,"commande passée pour une vente lacoste , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . pas pro , pas sérieux , aucun geste commercial hormis le remboursement promis des 2 articles manquants .",0,1,1
6,annulation de commande après 2 mois d ’ attente dans un geste et sans explication . retard de livraison et report à 3 reprise pour se résultat incompréhensible et inadmissible je recommande pas le site mieux vaut aller sur vente privée au mon d on attend mais on a sa commande,0,1,1
7,"extrêmement déçue de la vente apple ! ! achat d ’ un iphone reconditionné grade a+ ( fournisseur prs , phone recycle ) il a fonctionné 15 jours ! ! ! sav lamentable jusqu ’ a 6 jours pour répondre à une ’ question et pas de prise en charge des frais de retour pour réparation ... sav showroom qui lit les mails ’ en diagonale et répond à côté ! lamentable ... téléphone pas encore récupéré , ce sera mon dernier achat ! !",0,0,1
9,s'il y'avait une option : ne pas mettre d'étoile ( une étoile c'est trop ! ) je serais contente . j'ai passé une commande mi-avril et 

In [ ]:
# obligé de reprendre les données ici car bert n'a pas été entrainé sur dataset_final.pkl, donc j'avais un problème de matching entre les embeddings et les textes
df = pd.read_csv("../data/preprocessed/reviews_trust_clean.csv")
df_sans_contexte = pd.read_csv("../data/preprocessed/reviews_trust_clean_stopwords_supprimer.csv")

# on retire quelques stopwords supplémentaires
stopwords_custom = {
    'montre', 'montres', 'boucle', 'boucles', 'oreilles', 'oreille', 'paire', 'paires', 'lunettes', 'bracelet', 'bracelets', 'collier', 'iphone', 'téléphone', 'pendentif', 'robot', 'robots', 'aspirateur', 'baskets', 'basket', 'chaussure', 'chaussures', 'sandales', 'plantes', 'plante', 'arbres', 'arbre', 'bulbes', 'willemse', 'sommiers', 'jardin', 'bague', 'lampe', 'lampes', 'abat-jour', 'abat jour', 'lampadaire', 'parfum', 'shampoing', 'shampooing', 'shampooings', 'cheveux', 'masque', 'masques', 'crème', 'élastiques', 'manteau', 'bougie', 'cadre', 'écouteurs', 'vélo', 'robe', 'vêtements', 'bijoux', 'sac', 'portable', 'clio', 'luminaire', 'oreillette', 'induction', 'écouteur', 'couette', 'samsung', 'téléphones', 'smartcase', 'abat', 'apple', 'watch', 'shirt', 'tee', 'chemise', 'shirts', 'hortensias', 'orchidée', 'sacs', 'plant', 'chaises', 'lacoste', 'polo', 'pantalon', 'jean', 'jeans', 'sneakers', 'lunette', 'écran', 'tablette', 'tablettes', 'table', 'tables', 'chemisier', 'pulls', 'pull', 'trotinettes', 'trotinette', 'chaussons', 'chausson', 'brosse', 'brosses', 'crèmes', 'gel', 'gels', 'parfums', 'robe', 'robes', 'sacoches', 'sacoche', 'vestes', 'veste'
}

def clean_custom(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stopwords_custom]
    return " ".join(tokens)

df_sans_contexte["clean_comment"] = df_sans_contexte["clean_comment"].apply(clean_custom)
df["clean_comment"] = df["clean_comment"].apply(clean_custom)

# on conserve que les commentaires de plus de 100 caractères
mask = df["clean_comment"].str.len() > 100

df = df[mask]
df_sans_contexte = df_sans_contexte.loc[df.index]

texts_sans_contexte = df_sans_contexte["clean_comment"].astype(str).tolist()
texts_avec_contexte = df["clean_comment"].astype(str).tolist()

# # Si on modifie encore les stopwords et qu'on veut regénérer les embeddings, décommentez le code ci-dessous
# # --------- Pour regénerer les embeddings, décommentez le code ci-dessous ---------
model = SentenceTransformer('dangvantuan/french-document-embedding', trust_remote_code=True)
X_ctx = model.encode(
    texts_avec_contexte,
    batch_size=32,
    show_progress_bar=True
)
with open("../data/embeddings/emb_sbert_fr_ctm.pkl", "wb") as f:
    pickle.dump(X_ctx, f)
# # --------- Pour regénerer les embeddings, décommentez le code ci-dessus ---------


with open("../data/embeddings/emb_sbert_fr_ctm.pkl", "rb") as f:
    X_ctx = pickle.load(f)
    
X_ctx = np.asarray(X_ctx)
assert X_ctx.shape[0] == len(texts_avec_contexte), "Mismatch nb docs vs nb embeddings"

Batches:   0%|          | 0/259 [00:00<?, ?it/s]

In [21]:
tp = TopicModelDataPreparation("camembert-base")

training_dataset = tp.fit(
    text_for_contextual=texts_avec_contexte,
    text_for_bow=texts_sans_contexte,
    custom_embeddings=X_ctx
)

bow_size = len(tp.vocab)
ctx_size = X_ctx.shape[1]
bow_size, ctx_size

(20043, 768)

In [22]:
K = 30 # nombre de topics
ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=ctx_size,
    n_components=K,
    num_epochs=20,
    num_data_loader_workers=0
)

ctm.fit(training_dataset) # entraînement

Epoch: [20/20]	 Seen Samples: [165120/165580]	Train Loss: 382.26541942034584	Time: 0:00:02.289934: : 20it [00:46,  2.33s/it]
100%|██████████| 130/130 [00:01<00:00, 82.83it/s]


In [23]:
topics_words = ctm.get_topic_lists(20)  #  le nombre de mots à afficher par topic
for k, words in enumerate(topics_words):
    print(f"Topic {k}: {', '.join(words)}")

Topic 0: livreur, domicile, reçue, poste, recu, déposé, bureau, chercher, lettres, passage, laissé, journée, retirer, mains, quelqu, dpd, allée, dites, facteur, gls
Topic 1: endommagé, rétracter, madame, imaginer, falloir, droits, paquets, plateau, mention, nulle, satisfait, sent, courage, physique, abîmé, existait, revoir, laissée, recevrais, correctement
Topic 2: est, plus, qu, site, vente, remboursement, ai, service, être, fois, faire, client, fait, mail, avoir, depuis, privée, bien, après, retour
Topic 3: colis, mail, qu, service, remboursement, commande, jours, ai, après, relais, part, demande, chez, client, faire, jamais, donc, est, rien, avoir
Topic 4: tissus, tailles, taille, irl, qualité, agréable, recommanderai, collection, bottines, finitions, aime, grand, longues, dessus, pied, grandes, sortent, garde, rapides, connais
Topic 5: permets, 98, partager, matériel, existait, grandir, autorise, rendent, foutage, résumé, trouvent, membre, 2000, page, rythme, sommier, fonctionnera,

In [24]:
doc_topic = ctm.get_doc_topic_distribution(training_dataset)  # shape (n_docs, K)
df["topic_id"] = np.argmax(doc_topic, axis=1)
df["topic_confidence"] = doc_topic.max(axis=1)

100%|██████████| 130/130 [00:01<00:00, 80.34it/s]


In [13]:
os.makedirs("./artifacts/ctm", exist_ok=True)
os.makedirs("./artifacts/ctm/exports", exist_ok=True)

# ctm.save(models_dir="./artifacts/ctm/model")

# with open("./artifacts/ctm/vocab.pkl", "wb") as f:
#     pickle.dump(tp.vocab, f)

pd.DataFrame({
    "topic_id": np.arange(len(topics_words)),
    "top_words": [", ".join(w) for w in topics_words]
}).to_csv("./artifacts/ctm/exports/topics_top_words.csv", index=False)

np.save("./artifacts/ctm/exports/doc_topic.npy", doc_topic)

df.to_csv("./artifacts/ctm/exports/reviews_with_topics.csv", index=False)

In [25]:
topic_counts = df["topic_id"].value_counts().sort_index()
display(topic_counts)

topic_id
0     349
1     205
2      31
3      34
4     340
5      90
6     130
7     378
8     141
9     201
10    378
11    180
12    263
13     36
14    617
15    550
16    274
17    519
18    303
19     44
20    282
21    337
22    380
23    233
24    380
25    241
26    280
27     84
28    446
29    553
Name: count, dtype: int64

In [34]:
# Exemple de quelques commentaires par topic
def examples_for_topic(t, n=5):
    return df[df["topic_id"]==t]["clean_comment"].head(n).tolist()

for t in range(K):
    ex = examples_for_topic(t, 10)
    print(f"\n=== Topic {t} ===")
    for e in ex:
        print("-", e[:])


# 100 commentaires par catégories
# --- Définition des groupes de topics ---
# topics_livraison = [0, 3, 15, 17, 19, 20, 24, 27]
# topics_service_client = [1, 2, 6, 7, 8, 11, 14, 18, 21, 22, 23, 25, 28]
# topics_qualite_produit = [4, 9, 10, 12, 13, 16, 26, 29]

# # --- Fonction pour récupérer un nombre donné d'avis par catégorie ---
# def sample_by_topics(topic_list, n=100):
#     df_filtered = df[df["topic_id"].isin(topic_list)]
#     # si moins de 100 avis, on prend tout
#     return df_filtered["clean_comment"].sample(min(n, len(df_filtered)), random_state=42).tolist()

# # --- Récupération des 100 avis ---
# livraison_examples = sample_by_topics(topics_livraison, n=100)
# service_client_examples = sample_by_topics(topics_service_client, n=100)
# qualite_produit_examples = sample_by_topics(topics_qualite_produit, n=100)

# # --- Affichage propre ---
# def print_examples(label, examples):
#     print(f"\n=== {label} : {len(examples)} avis ===")
#     for e in examples:
#         print("-", e[:200])  # tronquer à 200 caractères pour lisibilité

# print_examples("SERVICE LIVRAISON", livraison_examples)
# print_examples("SERVICE CLIENT", service_client_examples)
# print_examples("QUALITÉ PRODUIT", qualite_produit_examples)



=== Topic 0 ===
- vente article manquant photo prise sur 6 articles la moitié livrée sans explication manque de respect.numero de commande 230130353
- commande passée pour une vente , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . pas pro , pas sérieux , aucun geste commercial hormis le remboursement promis des 2 articles manquants .
- ça fait un mois que ma commande aurait du être livré et showroom fait traîner l ’ affaire malgré mes appels et messages pourtant chronopost m ’ a indiqué le colis comme perdu
- dernièrement j'ai publié un avis mentionnant ma mauvaise expérience lors de l'achat d'un lot de boxers fila.après plusieurs échanges avec showroomprive , le services clients a finalement accepté que je retourne les articles . leur analyse a abouti à la décision de me rembourser.bravo showroomprive pour avoir accepté de revoir votre position .
- très deçu . j'ai commande une valise qui n'est jamais arrivée mais marquée comme

In [28]:
#Évaluer la diversité des topics
def topic_diversity(topic_words):
    """
    Calculate topic diversity: proportion of unique words among top words of all topics.
    topic_words : list of lists, each sublist contains the top words of a topic.
    """
    top_words = []
    for words in topic_words:
        top_words.extend(words)

    unique_words = set(top_words)
    diversity = len(unique_words) / len(top_words)
    return diversity
diversity_score = topic_diversity(topics_words)

print(f"Diversité des topics: {diversity_score:.2f}") # closer to 1 : each topic uses unique words(good diversity)

Diversité des topics: 0.66


In [29]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# texts : liste de documents, chaque document = liste de tokens
# topic_words : liste de listes des mots top de chaque topic
texts = df_sans_contexte["clean_comment"].apply(lambda x: x.split()).tolist()
dictionary = Dictionary(texts)

cm = CoherenceModel(
    topics=topics_words,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = cm.get_coherence()

print("Score de cohérence:", coherence_score)

Score de cohérence: 0.4528591754957337


In [30]:
for c in df[df["topic_id"] == 0]["Commentaire"]:
    print(c, "\n---\n")

Vente lacoste article manquant photo prise sur 6 articles la moitié livrée sans explication manque de respect.Numero de commande 230130353 
---

Commande passée pour une vente Lacoste , livraison 15 nous après la date prévue , déjà 6 semaines après l'achat , sur 3 produits , 2 manquants . Pas pro , pas sérieux , aucun geste commercial hormis le remboursement promis des 2 articles manquants . 
---

Ça fait un mois que ma commande aurait du être livré et Showroom fait traîner l ’ affaire malgré mes appels et messages Pourtant Chronopost m ’ a indiqué le colis comme perdu 
---

Dernièrement j'ai publié un avis mentionnant ma mauvaise expérience lors de l'achat d'un lot de boxers FILA.Après plusieurs échanges avec Showroomprive , le services clients a finalement accepté que je retourne les articles . Leur analyse a abouti à la décision de me rembourser.Bravo Showroomprive pour avoir accepté de revoir votre position . 
---

Très deçu . J'ai commande une valise qui n'est jamais arrivée mais 

In [ ]:
# Regroupement des topics
qualite_produit_topics = {3, 6, 18, 19, 23, 29}
livraison_topics = {5, 10, 24, 26, 27}
service_client_topics = {7, 8, 13, 14, 28}

df['label'] = np.nan

df.loc[df['topic_id'].isin(qualite_produit_topics), 'label'] = 0
df.loc[df['topic_id'].isin(livraison_topics), 'label'] = 1
df.loc[df['topic_id'].isin(service_client_topics), 'label'] = 2

df['label'] = df['label'].astype('Int64')  # int nullable
df['label'].value_counts(dropna=False)

label
<NA>    6191
0       4732
1       2374
2       1793
Name: count, dtype: Int64

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


              precision    recall  f1-score   support

         0.0      0.953     0.864     0.906       946
         1.0      0.796     0.819     0.807       475
         2.0      0.756     0.914     0.827       359

    accuracy                          0.862      1780
   macro avg      0.835     0.865     0.847      1780
weighted avg      0.871     0.862     0.864      1780



Classe prédite,0.0,1.0,2.0
Classe réelle,,,
0,817,76,53
1,33,389,53
2,7,24,328


,text,label_true,label_pred,correct
1,j'ai du leurs écrire pendant 2 mois pour me faire rembourser et en plis je n'ai reçu que la moitié . je n'ai jamais reçu ma commande et le service client vous prends pour des… je ne sais même pas s'il s'agit de réels personnes . je ne commanderai plus jamais .,0,2.0,False
3,sur 5 commande jai eu 3 problèmes soit produit manquant soit apres 2 mois d'attente 2 jours avant la livraison ma commande a etait annulé ... ou reception d'un autre article qui n'était pas le mien ... j'hésite a recommander ...,0,1.0,False
6,erreur de livraison . pieces manquantes . colis incomplet .,0,1.0,False
8,j ’ attends depuis des semaines le remboursement de 2 articles retournés ....,0,2.0,False
21,"commande livrée très vite ce qui est très agréable , car bien souvent les délais sont trop longs",0,1.0,False
39,"une catastrophe , le colis est arrivé ouvert et en mauvais état . les produits avaient fuit et étaient a même le carton . une honte .",1,0.0,False
40,"des années que je commande sur veepee et toujours au top . quand je commande chez eux je n'ai jamais été déçu , ils ne se trompent pas dans l'article comme shorommprive trois fois ils se sont trompé de commande . je reçois la plupart toujours à l'avance mais rarement en retard . merci",0,2.0,False
41,une commande passée il y a 4mois . je viens tout juste de recevoir un mail pour me dire que je vais être remboursé et qu ’ ils n ’ allaient pas me livrer ... un service déplorable . je déconseille fortement .,1,2.0,False
44,"commande effectuée le 17 décembre , soit disant expédiée le 20 . j ’ ai contacté le service client par e-mail et j ’ ai eu la chance d ’ obtenir une réponse type « ils effectuent des recherches et me tiennent au courant » . aujourd ’ hui , 2 janvier , aucune nouvelle de mon colis . mon compte en banque a bien sûr été débité depuis longtemps . je extrêmement déçue par l ’ évolution de ce site de vente en ligne .",1,2.0,False
50,"retard sur la livraison mais assez satisfaite sur le produit . seul regret , j ’ aurai du commander taille 54 au lieu de 52",0,1.0,False
